# YouDub · SenseVoice STT / Google Colab
在「运行时 → 更改运行时类型」选择 GPU。上传随附的 `youdub-stt-colab.zip`，不要上传 `.env`、Cookie 或 API Key。
此 notebook 只做 STT，不调用翻译或声音克隆服务。音视频会上传至 Google Colab。
输出 JSON 和原文 SRT；SRT 用于检查原文识别结果。现有网页 SRT 上传会跳过翻译，不要将原文 SRT 当译文上传。
只修复已有结果：上传新版源码包并安装依赖后，跳过音视频识别与其下载单元，直接运行最后的「重新分段」单元；无需 GPU。
分段默认上限为 4.5 秒 / 65 个原文字符，停顿阈值为 1 秒；跨 VAD 合并短碎片，保持较短的字幕阅读单位。
分段会优先选附近停顿，并平衡前后长度，减少孤立尾词。更新源码包后可直接用最后的重新分段单元处理已有 asr.json，无需重新识别。


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, sys, os, subprocess
bundle = files.upload()
assert len(bundle) == 1, "请选择一个 youdub-stt-colab.zip"
root = Path('/content/youdub-stt')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(bundle.values())))) as z:
    for info in z.infolist():
        target = (root / info.filename).resolve()
        assert target.is_relative_to(root.resolve()), "Invalid archive path"
    z.extractall(root)
del bundle
os.chdir(root)


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (resegment only)', 'PyTorch:', torch.__version__)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-stt.txt'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.DEVNULL)


上传一段音频或视频。优先使用分离后的人声 WAV；直接用原视频也能转录，但背景音乐可能影响质量。
模型首次运行需要下载。包版本固定为 FunASR 1.4.16；模型默认从 HF/ModelScope 下载，严格复现时请设置 FUNASR_MODEL / FUNASR_VAD_MODEL 为固定快照的本地目录。


In [ ]:
import uuid
uploaded = files.upload()
assert len(uploaded) == 1, "请选择一个音频或视频文件"
name, content = next(iter(uploaded.items()))
audio = root / ('input-' + uuid.uuid4().hex + Path(name).suffix)
audio.write_bytes(content)
del uploaded, content
language = 'en'  # en / zh / ja / ko / yue
output = root / ('output-' + uuid.uuid4().hex)
subprocess.run([sys.executable, 'scripts/run_stt.py', str(audio), '--language', language,
                '--device', 'cuda:0', '--output', str(output)], check=True)


In [ ]:
import shutil
result_zip = shutil.make_archive(str(output), 'zip', root_dir=output)
files.download(result_zip)
print((output / 'subtitles.srt').read_text(encoding='utf-8')[:1500])


## 可选：已有识别结果重新分段（无需重新推理）
先运行前面的新版源码上传和依赖安装单元。上传旧结果 ZIP 中的 `metadata/asr.json`，不要选 `asr_fixed.json`；保留原词级时间戳，输出新 ZIP。


In [ ]:
import uuid, shutil
previous = files.upload()
assert len(previous) == 1, "请选择原始 asr.json"
original = root / ('asr-original-' + uuid.uuid4().hex + '.json')
original.write_bytes(next(iter(previous.values())))
del previous
regrouped = root / ('regrouped-' + uuid.uuid4().hex)
subprocess.run([sys.executable, 'scripts/run_stt.py', str(original), '--resegment',
                '--output', str(regrouped)], check=True)
files.download(shutil.make_archive(str(regrouped), 'zip', root_dir=regrouped))
print((regrouped / 'subtitles.srt').read_text(encoding='utf-8')[:2000])
